In this script, a filter will be applied to each of the blast2go files, filtering the "Tags" column to retain only the rows with the "Under" label. Then, the resulting DataFrame will be sorted by the "P-Value" column. The top 5 rows from each blast2go file will be selected, and a single file will be created containing the top 5 rows from each file. If there are fewer than 5 top rows in any file, include as many as there are (1, 2, 3, or 4).

In [4]:
import os
import pandas as pd
import numpy as np

**Step 1**
Initial filtering to obtain the top 5 rows.

In [6]:
# Define the directory where the blast2go files are located
directory = "C:/Users/34698/OneDrive/Escritorio/Trabajos_2024/Factores de transcripcion/DAP/Picos/Pfam/"

# Get the list of files in the directory that contain "blast2go" in their name
blast2go_files =[f for f in os.listdir(directory) if "blast2go" in f]

# List to store the filtered dataframes from each file
filtered_dfs =[]

# Iterate over each blast2go file
for filename in blast2go_files:
    file_path= os.path.join(directory, filename)
    
    # Read the file
    df = pd.read_csv(file_path, sep="\t")
    
    # Filter the rows where the Tags column has "OVER"
    filtered_df = df[df["Tags"]== "[OVER]"]
    
    # Sort by the "P-Value" column
    filtered_df = filtered_df.sort_values(by="P-Value")
    
    # Select the first 5 rows
    top5_df =filtered_df.head(5)
    
    # Add to the general dataframe
    filtered_dfs.append(top5_df)
    
# Concatenate all filtered dataframes into one
final_df = pd.concat(filtered_dfs, ignore_index=True)

# Remove duplicates in the 'Annotation' column
final_df = final_df.drop_duplicates(subset=['Annotation'])
final_df = final_df["Annotation"]

# Define the name of the output file
output_filename = os.path.join(directory, "blast2go_top5_combined.txt")

# Save the final dataframe to a file
final_df.to_csv(output_filename, sep="\t", index=False)

print(f"Combined file saved: {output_filename}")


Combined file saved: C:/Users/34698/OneDrive/Escritorio/Trabajos_2024/Factores de transcripcion/DAP/Picos/Pfam/blast2go_top5_combined.txt


**Step 2**
Combine the previously created file, blast2go_top5_combined.txt, with each of the original blast2go files, ensuring that the rows from the resulting file are kept while checking if they have corresponding entries in each blast2go file. Then, retain the columns "Nr Test" and "Nr Reference" from each blast2go file and save them in separate files.

**Obtaining the dataframe with the observed and expected values of the transcription factor targets**

In [8]:
# Define the directory where the blast2go files are located
directory = "C:/Users/34698/OneDrive/Escritorio/Trabajos_2024/Factores de transcripcion/DAP/Picos/Pfam/"  

# Read the combined file
combined_file_path = os.path.join(directory, "blast2go_top5_combined.txt")
combined_df = pd.read_csv(combined_file_path, sep="\t")

# Get the list of files in the directory that contain "blast2go" in their name
blast2go_files = [f for f in os.listdir(directory) if "blast2go" in f]

# Iterate over each blast2go file
for filename in blast2go_files:
    file_path = os.path.join(directory, filename)
    
    # Read the original blast2go file
    blast2go_df = pd.read_csv(file_path, sep="\t")
    
    # Merge the combined file and the blast2go file on the 'Annotation' column
    merged_df = pd.merge(combined_df, blast2go_df, on='Annotation', how='left', suffixes=('', '_original'))
    
    # Print the available column names to verify
    # print(f"Columns in {filename}: {merged_df.columns.tolist()}")
    
    # Check if the columns exist before selecting them and performing the sum
    if 'Nr Reference' in merged_df.columns and 'Non Annot Test' in merged_df.columns:
        
        # target genes
        merged_df["Target Genes"] = merged_df["Nr Test"]
        
        # no target genes
        merged_df["No target Genes"] = merged_df["Nr Reference"]
        
        # Create a new column that is the sum of 'Nr Test' and 'Non Annot Test'
        merged_df['Total target'] = merged_df['Target Genes'] + merged_df['Non Annot Test']
        
        # Create a new column 'Total No target' which is 10700 - (Sum of Nr Reference + Non Annot Test)
        merged_df['Total No target'] = 10700 - merged_df['Total target']
        
        # Calculate the 'exp' column
        merged_df["exp"] = (merged_df["Total target"] / (merged_df["Total target"] + merged_df["Total No target"])) * (merged_df["Target Genes"] + merged_df["No target Genes"])
        
        # Calculate the 'ratio' column
       
        # Calculate the 'ratio' column and handle cases where 'exp' is zero
        merged_df["ratio"] = merged_df["Target Genes"] / merged_df["exp"]
        
        merged_df.loc[merged_df["Target Genes"] == 0, "ratio"] = np.nan
        # Calculate the 'log2' column using log base 2, setting 0 if ratio is 0 or NaN
        merged_df["log2"] = np.where(merged_df["ratio"].isna(), 0, np.log2(merged_df["ratio"]))

        
        # Select the final columns
        result_df = merged_df[['Annotation', 'Target Genes', 'No target Genes', 'Total target', 'Total No target', "exp", "ratio", "log2"]]
        
        # Define the name of the output file
        output_filename = os.path.join(directory, f"{filename.replace('.txt', '_result.txt')}")
        
        # Save the filtered dataframe to a file
        result_df.to_csv(output_filename, sep="\t", index=False)
        
        print(f"Result file saved: {output_filename}")
    else:
        print(f"Columns 'Nr Reference' and 'Non Annot Test' not found in {filename}.")


Result file saved: C:/Users/34698/OneDrive/Escritorio/Trabajos_2024/Factores de transcripcion/DAP/Picos/Pfam/blast2go_table_bzip2_result.txt
Result file saved: C:/Users/34698/OneDrive/Escritorio/Trabajos_2024/Factores de transcripcion/DAP/Picos/Pfam/blast2go_table_bZip3_result.txt
Result file saved: C:/Users/34698/OneDrive/Escritorio/Trabajos_2024/Factores de transcripcion/DAP/Picos/Pfam/blast2go_table_bZip5_result.txt
Result file saved: C:/Users/34698/OneDrive/Escritorio/Trabajos_2024/Factores de transcripcion/DAP/Picos/Pfam/blast2go_table_bZIP6_result.txt
Result file saved: C:/Users/34698/OneDrive/Escritorio/Trabajos_2024/Factores de transcripcion/DAP/Picos/Pfam/blast2go_table_bZip7_result.txt
Result file saved: C:/Users/34698/OneDrive/Escritorio/Trabajos_2024/Factores de transcripcion/DAP/Picos/Pfam/blast2go_table_CopF3_result.txt
Result file saved: C:/Users/34698/OneDrive/Escritorio/Trabajos_2024/Factores de transcripcion/DAP/Picos/Pfam/blast2go_table_CP2_result.txt
Result file sav

**Obtaining the matrix with the enrichment p-values of the transcription factor targets**

In [9]:
# Define the directory where the result files are located
directory = "C:/Users/34698/OneDrive/Escritorio/Trabajos_2024/Factores de transcripcion/DAP/Picos/Pfam/"

# Get the list of files in the directory that contain "_result.txt" in their name
result_files = [f for f in os.listdir(directory) if "_result.txt" in f]

# Initialize a dictionary to store the log2 data
log2_data = {}

# Iterate over each result file
for filename in result_files:
    file_path = os.path.join(directory, filename)
    
    # Read the result file
    result_df = pd.read_csv(file_path, sep="\t")
    
    # Extract the column name for the file
    col_name = filename.split('_')[2]  # Assuming the file name is like 'blast2go_table_bzip2_result.txt'
    
    # Store the 'log2' column in the dictionary with the extracted name
    log2_data[col_name] = result_df.set_index('Annotation')['log2']

# Convert the dictionary into a DataFrame
log2_matrix = pd.DataFrame(log2_data)

# Add the 'Annotation' column to the DataFrame
log2_matrix.insert(0, 'Annotation', log2_matrix.index)

# Save the resulting DataFrame into a file
output_matrix_file = os.path.join(directory, "log2_summary_matrix.txt")
log2_matrix.to_csv(output_matrix_file, sep="\t", index=False)

print(f"Log2 matrix saved in: {output_matrix_file}")


Log2 matrix saved in: C:/Users/34698/OneDrive/Escritorio/Trabajos_2024/Factores de transcripcion/DAP/Picos/Pfam/log2_summary_matrix.txt
